# 📅 2026-03-23 (월) 개발 노트 : 풀스택 기본 뼈대 구축 및 AI 추천 데이터 파이프라인 연동

## 🎯 오늘의 목표
- [x] 프론트엔드(Next.js) 환경 복구 및 Tailwind CSS 디자인 시스템 연동
- [x] 백엔드(FastAPI) 추천 API 로직 구축 및 통신 허용(CORS) 설정
- [x] 프론트엔드와 백엔드 통신 연결 및 검색 결과(게임 데이터) 화면 렌더링 성공
- [x] 추천 시스템 핵심 메타데이터 수집 파이프라인 (RAWG API) 구축
- [x] LLM(Upstage Solar) 기반 12가지 게임 평가 지표 자동 추출 로직 구현
- [x] Vector DB 도입을 위한 Sentence-Transformers 텍스트 임베딩 및 유사도 검색 기초 테스트

## 🛠 진행 상황 및 핵심 기록
- `package.json` 수동 생성 및 `npm install`을 통한 프론트엔드 엔진/의존성 패키지 전면 복구 완료.
- `next.config.mjs` 확장자 변경 및 `layout.tsx` 폰트 충돌 에러(Geist -> Inter) 해결.
- Tailwind CSS 3총사(tailwindcss, postcss, autoprefixer) 설치 및 `tailwind.config.js`, `globals.css` 셋업으로 UI 디자인 완벽 적용.
- **백엔드 API 엔드포인트:** `GET http://localhost:8000/recommend?query={검색어}` 
  - 검색어(대소문자 무시)를 기반으로 게임 이름을 필터링하고 임시 `final_score`를 계산하는 로직 적용.
- **프론트엔드-백엔드 연동:** 프론트엔드의 `fetch` 함수를 통해 백엔드 데이터를 JSON으로 받아와 `results` 상태값을 업데이트하고 카드 형태로 출력함.
- **데이터 수집 파이프라인 (Data Pipeline):** RAWG API를 활용해 10개 주요 장르, 1,000개 이상의 게임 데이터(이름, 장르, 상세 설명 등) 수집. 결측치 및 너무 짧은 설명(100자 이하)을 걸러내는 퀄리티 컨트롤을 거쳐 최종 583개의 정예 데이터 확보.
- **LLM 데이터 가공 (Data Enrichment):** Upstage Solar 모델(`solar-mini`)을 연동. 시스템 프롬프트(Persona) 설정 및 반복문을 통해 `mania_score`, `story_depth` 등 12개 커스텀 지표를 추출하고 20개 단위로 자동 저장하는 Fail-safe 파이프라인 구축.
- **임베딩 및 벡터 검색 테스트:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` 다국어 모델을 로드하여 텍스트를 고차원 벡터로 변환하고, `cosine_similarity`를 통해 유저 검색어와 게임 설명 간의 의미적 유사도 검색 로직 검증 완료.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제 1: 프론트엔드 서버 실행 시 화면에 디자인(Tailwind)이 적용되지 않고 깨지는 현상**
  - **원인:** CSS 엔진에 스타일을 맵핑할 `tailwind.config.js` 설정 파일이 누락되어 브라우저가 클래스명을 인식하지 못함.
  - **해결:** `npx tailwindcss init -p` 명령어로 설정 파일을 생성하고, `globals.css` 상단에 `@tailwind` 디렉티브 3줄을 추가하여 엔진을 연결함.

- **문제 2: 검색 버튼 클릭 시 무반응 및 백엔드에 신호가 가지 않음**
  - **원인:** 프론트엔드에서 요청할 `/recommend` API 창구가 백엔드에 없었으며, 포트가 다른 두 서버 간의 통신을 브라우저가 보안(CORS) 이유로 차단함.
  - **해결:** `main.py`에 `@app.get("/recommend")` 라우터를 추가하고, `CORSMiddleware`를 도입해 `allow_origins=["*"]`로 통신 문을 개방함. (주소도 `127.0.0.1`에서 `localhost`로 통일)

- **문제 3: 영어로 정상 검색 시 백엔드에서 500 Internal Server Error (`KeyError: 'added'`) 발생**
  - **원인:** 백엔드 점수 계산 로직에서 `added`, `rating` 데이터를 찾았으나, 실제 로드한 CSV 파일 내부에 해당 컬럼이 누락되어 있어 파이썬 판다스 엔진이 뻗어버림.
  - **해결:** 데이터 존재 여부를 먼저 확인하고, 값이 없을 경우 `.fillna(0)`을 통해 0점 처리하도록 백엔드 방탄(안전장치) 코드를 추가하여 런타임 에러를 원천 차단함.

- **문제 4: 로컬 환경 차이로 인한 파일 경로 에러 (`FileNotFoundError` 및 `df is not defined`)**
  - **원인:** 하드코딩된 절대 경로(`C:\...`) 사용 및 주피터 노트북의 실행 폴더 깊이 차이로 인해 프로그램이 데이터를 찾지 못함.
  - **해결:** 하드코딩을 폐기하고, `.env` 파일의 위치를 등대처럼 삼는 `find_dotenv()`와 `os.path.dirname()`을 조합. `os.path.join()`을 활용해 어떤 PC에서 실행하든 동적으로 프로젝트 최상위 루트를 추적하는 협업용 경로 탐색 로직 적용.

- **문제 5: AI가 숫자 외의 텍스트를 반환하여 형변환 에러 (`invalid literal for int()`) 발생**
  - **원인:** LLM에게 '숫자만 출력하라'고 지시해도 "8점입니다"와 같이 불필요한 텍스트를 섞어 반환하는 할루시네이션(Hallucination) 발생.
  - **해결:** 파이썬 정규표현식(`re.findall(r'\d+', ai_reply)`)을 장착하여 문자열 내에서 연속된 숫자만 강제 추출하도록 전처리함. 또한 점수 범위를 제한하고 오류 시 기본값(5)을 부여하는 방어 로직을 추가하여 대규모 루프 중단을 방지함.

## 💡 인사이트 및 다음 할 일
- **배운 점:** 프론트엔드와 백엔드를 연결할 때는 **CORS 설정**이 필수임을 체감함. 또한, AI 모델은 언제든 예상치 못한 형태의 텍스트를 뱉을 수 있으므로 **정규표현식을 활용한 출력값 통제**가 필수적임. 추천 시스템에 있어 데이터의 양보다 철저한 정제가 들어간 '질(Quality)'이 중요함을 깨달았으며, 개발 전반에서 파일 경로 하드코딩은 유지보수를 해치는 가장 큰 적임을 체득함.
- **다음 할 일:** 1. API 고도화: 게임 카드 UI에 RAWG API를 연결하여 진짜 게임 포스터 이미지 띄우기.
  2. 추천 로직 진화: 오늘 Jupyter에서 검증한 Sentence-Transformers 임베딩 및 유사도 측정 로직을 PostgreSQL(pgvector) Vector DB에 적재하고, FastAPI 백엔드에 통합하여 진짜 'AI 추천 검색' 파이프라인 완성하기.